# Reporting and Visualisation 

The notebook has data reporting and visualisation, for the IOT555U Project. 

SQL-based analysis and visual outputs from the relational database made in synthetic_data_gen. Patient flow, left before being seen , triage demands in an A&E department

## Setup

In [ ]:
import psycopg2 
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import os
from dotenv import load_dotenv

In [2]:
# Database Connections
load_dotenv()
conn_string = os.getenv('NEON_URL')

conn = psycopg2.connect(conn_string)
cur = conn.cursor()

cur.execute("SELECT version();" )
print(cur.fetchone())

python-dotenv could not parse statement starting at line 7


('PostgreSQL 17.8 (92d3c18) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit',)


## Analysis 

### Patient Flow

In [ ]:
# Picks the triage name, calculates the average number of minutes from x timestamp to y timesamp

query_1 = '''
SELECT
    t.triage_name,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.first_assessment_time - v.arrival_time)) / 60), 2) AS avg_mins_to_first_assessment,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.treatment_start - v.arrival_time)) / 60), 2) AS avg_mins_to_treatment_start,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 60), 2) AS avg_total_visit_mins,
    COUNT(*) AS visit_count
FROM visit v
JOIN triage_category t
    ON v.triage_id = t.triage_id
WHERE v.first_assessment_time IS NOT NULL
GROUP BY t.triage_name
ORDER BY avg_mins_to_first_assessment DESC;
'''

query_1_df = pd.read_sql(query_1, conn)
query_1_df

/var/folders/5y/20qzsxn55txdbxf0jcdh23h00000gn/T/ipykernel_67833/2588995143.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_1_df = pd.read_sql(query_1, conn)


,triage_name,avg_mins_to_first_assessment,avg_mins_to_treatment_start,avg_total_visit_mins,visit_count
0,Not Urgent,19.04,85.72,239.99,28640
1,Not Threatening to Life & Limb,13.22,81.02,239.17,78699
2,Urgent,9.28,77.81,237.30,35952
3,Very Urgent,4.27,75.43,238.30,756
4,Life Threatening Conditions,1.76,61.04,175.48,25


In [ ]:
fig = go.Figure()

# Stages of a patient flow 


fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_mins_to_first_assessment'],
    name='First Assessment',
    marker_color='lightslategray', 
    opacity=0.4
))

fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_mins_to_treatment_start'] - query_1_df['avg_mins_to_first_assessment'],
    name='Assessment to Treatment',
    marker_color='slategrey', 
    opacity=0.7
))

fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_total_visit_mins'] - query_1_df['avg_mins_to_treatment_start'],
    name='Treatment to Discharge',
    marker_color='darkslategrey',
    opacity=0.9
))

fig.update_layout(
    title='<b>Average Patient Journey Duration</b>',
    xaxis_title='Triage Category',
    yaxis_title='Minutes from Arrival',
    barmode='stack',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=100)
)

fig.show()

### Left before being seen

In [5]:
# Picks the triage name, calculates the average number of minutes from x timestamp to y timesamp. E

query_2 = '''
SELECT 
    t.triage_name,
    COUNT(*) AS total_patients,
    COUNT(CASE WHEN v.treatment_start IS NULL THEN 1 END) AS left_before_treatment,
    ROUND(100.0 * COUNT(CASE WHEN v.treatment_start IS NULL THEN 1 END) / COUNT(*), 1) AS pct_left_early
FROM visit v
JOIN triage_category t ON v.triage_id = t.triage_id
GROUP BY t.triage_name
ORDER BY pct_left_early DESC;
'''

query_2_df = pd.read_sql(query_2, conn)
query_2_df

/var/folders/5y/20qzsxn55txdbxf0jcdh23h00000gn/T/ipykernel_67833/18065189.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_2_df = pd.read_sql(query_2, conn)


,triage_name,total_patients,left_before_treatment,pct_left_early
0,Life Threatening Conditions,25,1,4.0
1,Not Urgent,28640,632,2.2
2,Urgent,35952,774,2.2
3,Not Threatening to Life & Limb,78699,1686,2.1
4,Very Urgent,756,11,1.5


In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Total patient volume
fig.add_trace(
    go.Bar(
        x=query_2_df['triage_name'],
        y=query_2_df['total_patients'],
        name="Total Patients",
        marker_color='lightslategray',
        opacity=0.6,
        hovertemplate="Category: %{x}<br>Total Patients: %{y}<extra></extra>"
    ),
    secondary_y=False,
)

#  percentage who left early
fig.add_trace(
    go.Scatter(
        x=query_2_df['triage_name'],
        y=query_2_df['pct_left_early'],
        name="% Left Before Treatment",
        mode='lines+markers',
        marker=dict(size=10, color='crimson'),
        line=dict(width=3),
        hovertemplate="Category: %{x}<br>Left Early: %{y}%<extra></extra>"
    ),
    secondary_y=True,
)

fig.update_layout(
    title='<b>Patients -  Left Before Treatment by Triage Category</b>',
    xaxis_title='Triage Category',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template='plotly_white',
    hovermode="x unified"
)

fig.update_yaxes(title_text="Number of Patients", secondary_y=False)
fig.update_yaxes(title_text="Percentage (%) Left Early", secondary_y=True, range=[0, query_2_df['pct_left_early'].max() + 5])

fig.show()

### Triage Demand

In [7]:
query_3 = '''
SELECT
    t.triage_name,
    COUNT(*) AS visit_count,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 60), 1) AS avg_total_visit_mins,
    ROUND(SUM(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 3600), 1) AS total_department_hours
FROM visit v
JOIN triage_category t
    ON v.triage_id = t.triage_id
WHERE v.discharge_time IS NOT NULL
GROUP BY t.triage_name
ORDER BY total_department_hours DESC;
'''

query_3_df = pd.read_sql(query_3, conn)
query_3_df

/var/folders/5y/20qzsxn55txdbxf0jcdh23h00000gn/T/ipykernel_67833/1585571180.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_3_df = pd.read_sql(query_3, conn)


,triage_name,visit_count,avg_total_visit_mins,total_department_hours
0,Not Threatening to Life & Limb,78699,239.2,313701.9
1,Urgent,35952,237.3,142191.0
2,Not Urgent,28640,240.0,114553.7
3,Very Urgent,756,238.3,3002.6
4,Life Threatening Conditions,25,175.5,73.1


In [ ]:
fig = px.treemap(
    query_3_df, 
    path=[px.Constant('Total A&E Workload'), 'triage_name'], 
    values='total_department_hours',
    color='total_department_hours',
    color_continuous_scale='Greys', 
    title='<b>Workload Distribution (Hours)</b>',
    template='plotly_white',
)

fig.update_traces(
    textinfo='label+value+percent parent',
    texttemplate='<b>%{label}</b><br>%{value:.1f} Hours<br>%{percentParent:.1%}',
    marker=dict(line=dict(width=1, color='white')) 
)

fig.update_layout(
    coloraxis_showscale=True,
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()